In [1]:
from langgraph.graph import StateGraph
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from typing import TypedDict
from langgraph.constants import END, START

In [2]:
load_dotenv()

True

In [3]:
class Blog(TypedDict):

    title: str
    outline: str
    content: str

In [4]:
llm = HuggingFacePipeline.from_model_id(
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    task="text-generation",
    pipeline_kwargs=dict(
        max_new_tokens=256,
        temperature=0.1,
        do_sample=True,
    )
)
model = ChatHuggingFace(llm=llm)

/Users/harshitjadiya/Desktop/LangGraph/myenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 22009.01it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [5]:
def outline(state: Blog) -> Blog:

    title = state["title"]

    prompt = f"Generate me a detailed outline for a blog on the topic {title}."

    outline = model.invoke(prompt).content

    state['outline'] = outline

    return state

In [6]:
def content(state: Blog) -> Blog:

    title = state['title']
    outline = state["outline"]

    prompt = f"Generate me a detailed blog on {title} using the outline {outline}."

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [7]:
graph = StateGraph(Blog)

graph.add_node('outline', outline)
graph.add_node('content', content)

graph.add_edge(START, "outline")
graph.add_edge("outline", "content")
graph.add_edge("content", END)

workflow = graph.compile()

In [ ]:
final = workflow.invoke({'title': "Rise of AI in India"})
print(final['content'])

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
